# Supplement · Interactive Maps with folium

**A short primer, so the maps in notebooks 05 and 06 are not a black box.**

Notebooks 05 and 06 draw the Sage nodes on an interactive map. Those maps are made
with folium, and if you have not seen folium before, the calls there can look like
magic. This supplement teaches folium from the ground up and finishes by building
the exact `makeNodeMap` function the capstone uses, so nothing in the main
notebooks is left unexplained.

Nothing here needs a live Sage query. Every map is built from a handful of fixed
Chicago coordinates, so the whole notebook runs offline. The background map tiles
do load from the internet when you open a map in a browser, so if the tiles are
blank, that is a network issue, not a code issue. The markers are computed locally
and always appear.

## 1. What folium is, and what a web map is made of

folium is a Python library that builds interactive "slippy" maps, the kind you can
pan and zoom in a browser. Under the hood it writes out a small web page powered by
Leaflet, a JavaScript mapping library. You never write JavaScript; you describe the
map in Python and folium produces the page.

A folium map has three kinds of parts.

- **Tiles**: the background image of the world, served as many small square images
  from a tile provider. We use OpenStreetMap tiles, which are free.
- **Markers**: the things you place on top, such as a pin or a colored circle at a
  latitude and longitude.
- **Popups and tooltips**: the text that appears when you click or hover a marker.

For a sensor network this is a natural fit: each node has a latitude and longitude,
so each node becomes a marker, and you can color the markers by a measured value.

In [ ]:
def ensureFolium():
    """
    Import folium and branca, installing folium into this kernel if it is missing.

    Uses sys.executable so the install targets the exact environment behind this
    kernel, and a normal non-editable install so the import works without a
    restart. Returns (foliumModule, colormapModule), or (None, None) if the
    install could not complete (for example, no network).
    """
    try:
        import folium
        import branca.colormap as colormap
        return folium, colormap
    except ImportError:
        import sys, subprocess, importlib
        print("folium not found, installing it into this kernel...")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "folium"],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            lastLine = (result.stderr.strip().splitlines() or ["unknown error"])[-1]
            print("folium install failed:", lastLine)
            return None, None
        importlib.invalidate_caches()
        import folium
        import branca.colormap as colormap
        return folium, colormap


import pandas as pd
from IPython.display import display

folium, cm = ensureFolium()
haveFolium = folium is not None
print("folium ready:", haveFolium)

## 2. Your first map

The starting point is `folium.Map`. Three arguments do most of the work.

- `location`: the center of the map, given as `[latitude, longitude]`.
- `zoom_start`: the initial zoom level. Small numbers show a whole region; larger
  numbers zoom in. Around 10 to 12 is a good city scale.
- `tiles`: the background style. `"OpenStreetMap"` is the free default.

Displaying the map object renders it inline. Try panning and zooming it.

In [ ]:
# Chicago, roughly centered on the Loop.
firstMap = folium.Map(location=[41.88, -87.63], zoom_start=11, tiles="OpenStreetMap")
display(firstMap)

## 3. Adding markers

Two marker types cover almost everything.

- `folium.Marker` is the classic teardrop pin. Good for a small number of labeled
  points.
- `folium.CircleMarker` is a circle whose size and color you control. Better when
  you have many points or want to encode a value in the color, which is exactly
  what the node maps do.

Every marker is created, then attached to a map with `.add_to(theMap)`. A `popup`
appears on click; a `tooltip` appears on hover.

In [ ]:
markerMap = folium.Map(location=[41.83, -87.62], zoom_start=12, tiles="OpenStreetMap")

# A classic pin with a click popup and a hover tooltip.
folium.Marker(
    location=[41.87, -87.65],
    popup="W06C, a Sage node",
    tooltip="hover shows this, click shows the popup",
).add_to(markerMap)

# A circle marker, sized and colored explicitly.
folium.CircleMarker(
    location=[41.79, -87.60],
    radius=10,
    color="crimson",
    fill=True,
    fill_color="crimson",
    fill_opacity=0.7,
    popup="W020",
).add_to(markerMap)

display(markerMap)

## 4. Placing several nodes and framing them

For a set of nodes, loop over their coordinates and add one marker each. Then, so
you do not have to guess the right center and zoom, let folium fit the view to the
points with `fit_bounds`, which takes a list of `[latitude, longitude]` pairs.

The dictionary below is the same set of Chicago nodes the capstone uses offline.

In [ ]:
nodeSites = {
    "W06C": {"lat": 41.87, "lon": -87.65, "meanF": 69.3},
    "W020": {"lat": 41.79, "lon": -87.60, "meanF": 72.9},
    "W039": {"lat": 42.01, "lon": -87.70, "meanF": 112.5},   # a warmer, odd node
}

multiMap = folium.Map(location=[41.9, -87.65], zoom_start=10, tiles="OpenStreetMap")

points = []
for vsn, site in nodeSites.items():
    folium.CircleMarker(
        location=[site["lat"], site["lon"]],
        radius=8,
        color="royalblue",
        fill=True,
        fill_color="royalblue",
        fill_opacity=0.8,
        popup=vsn,
    ).add_to(multiMap)
    points.append([site["lat"], site["lon"]])

multiMap.fit_bounds(points)   # zoom and center so every marker is visible
display(multiMap)

## 5. Coloring markers by a value

The point of the node maps is to see a measurement at a glance. To turn a number
into a color you need a color scale. branca, which ships with folium, provides
`LinearColormap`: you give it a list of colors and the low and high ends of your
value range, and calling it with a value returns a color.

Add the scale to the map with `map.add_child(scale)` and it appears as a legend.
Then color each marker by passing its value through the scale.

In [ ]:
valueMap = folium.Map(location=[41.9, -87.65], zoom_start=10, tiles="OpenStreetMap")

means = [site["meanF"] for site in nodeSites.values()]
scale = cm.LinearColormap(
    ["blue", "green", "yellow", "red"],
    vmin=min(means),
    vmax=max(means),
)
scale.caption = "node mean temperature (F)"
valueMap.add_child(scale)   # draws the color legend

for vsn, site in nodeSites.items():
    markerColor = scale(site["meanF"])   # value -> color
    folium.CircleMarker(
        location=[site["lat"], site["lon"]],
        radius=9,
        color=markerColor,
        fill=True,
        fill_color=markerColor,
        fill_opacity=0.85,
        popup=f"{vsn}: {site['meanF']} F",
    ).add_to(valueMap)

display(valueMap)

## 6. Map styles (basemaps)

Every map so far used the default OpenStreetMap basemap, a detailed street map,
sometimes called a roadmap. folium ships a few other named styles, and you can add
satellite imagery from a custom tile source. The right style depends on the job: a
street map for orientation, a light or dark minimal map when the data should
dominate, and satellite when you want to see the actual ground around a node.

The named styles that work out of the box:

- `OpenStreetMap`: the default street map.
- `CartoDB positron`: a light, low-ink map that lets colored markers stand out.
- `CartoDB dark_matter`: a dark version, good for bright markers.
- `CartoDB voyager`: a softer street map.

Satellite imagery is not built in, but you can point folium at a public tile
service. Esri's World Imagery is a common choice: you pass its tile URL and an
attribution string. A tile URL contains `{z}`, `{x}`, and `{y}` placeholders that
the map fills in as you pan and zoom.

In [ ]:
# The satellite tile source used throughout, plus its required attribution.
esriSatelliteTiles = ("https://server.arcgisonline.com/ArcGIS/rest/services/"
                      "World_Imagery/MapServer/tile/{z}/{y}/{x}")
esriAttribution = "Tiles (c) Esri, Maxar, Earthstar Geographics"

styleExamples = [
    ("OpenStreetMap", None, "roadmap"),
    ("CartoDB positron", None, "light"),
    ("CartoDB dark_matter", None, "dark"),
    (esriSatelliteTiles, esriAttribution, "satellite"),
]

for tiles, attr, label in styleExamples:
    styleMap = folium.Map(location=[41.87, -87.65], zoom_start=12,
                          tiles=tiles, attr=attr)
    folium.CircleMarker(
        location=[41.87, -87.65], radius=8, color="red",
        fill=True, fill_color="red", fill_opacity=0.9,
        popup=f"style: {label}").add_to(styleMap)
    print(f"style: {label}")
    display(styleMap)

### Let the viewer switch styles

Showing four separate maps is fine for a tour, but in practice it is friendlier to
put several styles on one map and add a `LayerControl`, which draws a switcher in
the corner so the viewer picks the basemap they want. Build the map with
`tiles=None`, add each style as a named `TileLayer`, then add the control.

In [ ]:
switchMap = folium.Map(location=[41.87, -87.65], zoom_start=11, tiles=None)

folium.TileLayer("OpenStreetMap", name="Roadmap").add_to(switchMap)
folium.TileLayer("CartoDB positron", name="Light").add_to(switchMap)
folium.TileLayer("CartoDB dark_matter", name="Dark").add_to(switchMap)
folium.TileLayer(esriSatelliteTiles, attr=esriAttribution, name="Satellite").add_to(switchMap)

folium.CircleMarker(
    location=[41.87, -87.65], radius=8, color="yellow",
    fill=True, fill_color="yellow", fill_opacity=0.9,
    popup="W06C").add_to(switchMap)

folium.LayerControl().add_to(switchMap)   # the style switcher, top right corner
display(switchMap)

## 7. The capstone's `makeNodeMap`, explained

Everything so far combines into the function the capstone calls. `makeNodeMap`
takes a dictionary of node locations and, optionally, a dictionary of values. When
values are given it colors the markers on a scale and, going one step further,
outlines in red any node whose value is far from the group. "Far" is measured with
the modified z-score: a node is flagged when its distance from the median of the
values exceeds 3.5 times the scaled median absolute deviation, the same robust rule
the analysis notebooks use for outliers.

Read it once here and the map cells in notebook 06 will be obvious.

In [ ]:
def makeNodeMap(locations, nodeValues=None, caption="node mean temperature (F)",
                tiles="OpenStreetMap", attr=None):
    """
    Build a folium map of node locations.

    Args:
        locations: {vsn: (lat, lon)} for each node to place.
        nodeValues: optional {vsn: number}. When given, markers are colored on a
            scale and any node far from the group is outlined in red.
        caption: the color legend caption.
        tiles: the basemap style, a named style or a tile URL. Pass
            esriSatelliteTiles for satellite imagery.
        attr: attribution string, required when tiles is a custom URL.

    Returns:
        A folium.Map, which renders inline in Jupyter, or None if locations is empty.
    """
    if not locations:
        print("no located nodes to map")
        return None

    lats = [lat for lat, _ in locations.values()]
    lons = [lon for _, lon in locations.values()]
    center = [sum(lats) / len(lats), sum(lons) / len(lons)]
    fmap = folium.Map(location=center, zoom_start=10, tiles=tiles, attr=attr)

    scale = None
    groupMedian = groupMad = None
    if nodeValues:
        values = [v for v in nodeValues.values() if v is not None]
        if values:
            scale = cm.LinearColormap(["blue", "green", "yellow", "red"],
                                      vmin=min(values), vmax=max(values))
            scale.caption = caption
            fmap.add_child(scale)
            valueSeries = pd.Series(values)
            groupMedian = valueSeries.median()
            groupMad = (valueSeries - groupMedian).abs().median()

    for vsn, (lat, lon) in locations.items():
        value = nodeValues.get(vsn) if nodeValues else None
        if value is None or scale is None:
            fillColor = "#3186cc"
            outline = "#3186cc"
            radius = 6
            popup = f"{vsn}"
        else:
            fillColor = scale(value)
            isOutlier = (groupMad is not None and groupMad > 0
                         and abs(value - groupMedian) > 3 * 1.4826 * groupMad)
            outline = "red" if isOutlier else fillColor
            radius = 10 if isOutlier else 7
            flag = "  (disagrees with the group)" if isOutlier else ""
            popup = f"{vsn}: {value:.1f} F{flag}"
        folium.CircleMarker(
            location=[lat, lon], radius=radius, color=outline, weight=3,
            fill=True, fill_color=fillColor, fill_opacity=0.85, popup=popup,
        ).add_to(fmap)

    return fmap


# Build the same map the capstone builds, from the node set above.
locations = {vsn: (site["lat"], site["lon"]) for vsn, site in nodeSites.items()}
values = {vsn: site["meanF"] for vsn, site in nodeSites.items()}

# The default roadmap version.
display(makeNodeMap(locations, nodeValues=values))

# The same node map on satellite imagery, which shows the ground around each site.
display(makeNodeMap(locations, nodeValues=values,
                    tiles=esriSatelliteTiles, attr=esriAttribution))

The warm node (W039 at 112.5 F) sits far above the other two, so it is outlined in
red and drawn larger. That is the same signal the capstone uses to flag a
miscalibrated or oddly sited node.

## 8. Saving a map to share

A folium map is just a web page, so you can write it to an HTML file and open it in
any browser or email it to someone. No Python is needed to view it.

In [ ]:
sharedMap = makeNodeMap(locations, nodeValues=values)
sharedMap.save("chicago_nodes.html")
print("saved chicago_nodes.html; open it in a browser to view the interactive map")

## 9. Exercises

1. Change `zoom_start` in the first map from 11 to 9 and then 13. What does each
   zoom level show, and when would you want each?
2. Add a fourth node of your own to `nodeSites` with a `meanF` between the others,
   then re-run section 6. Where does it land on the color scale?
3. Give your new node a `meanF` far above the rest. Does `makeNodeMap` outline it
   in red? Now nudge it closer to the group until the red outline disappears, and
   note the value where it flips.
4. Swap the color list in the `LinearColormap` for a different set, for example
   `["#2b83ba", "#ffffbf", "#d7191c"]`. How does the legend change?
5. Open the saved `chicago_nodes.html` in a browser. Do the tiles load? If not,
   what does that tell you about the difference between building a map and
   displaying it?
6. Rebuild the node map with `tiles=esriSatelliteTiles` and `attr=esriAttribution`.
   Do the colored markers read as clearly on satellite as on the roadmap? When
   would satellite be the better choice, and when would the light basemap win?

## Further reading

- folium documentation: https://python-visualization.github.io/folium/
- branca, the color scales folium uses: https://python-visualization.github.io/branca/
- Leaflet, the JavaScript library folium wraps: https://leafletjs.com/
- OpenStreetMap, the default map tiles: https://www.openstreetmap.org/
- A gallery of tile styles you can use with folium: https://leaflet-extras.github.io/leaflet-providers/preview/
- Esri World Imagery, the satellite source: https://www.arcgis.com/home/item.html?id=10df2279f9684e4a9f6a7f08febac2a9